## Exploración Inicial de Datos

### By:
Karen Ninco

### Date:
2026-08-20

### Description:

# Requerimiento

Crear una nueva rama de git (Usar [Gitflow](https://joserzapata.github.io/courses/ciencia-datos-en-produccion/control-versiones/branching-model/)) y Crear un notebook  para la exploración inicial de los datos

- Tomar como ejemplo los pasos de: <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/>

el objetivo es realizar una exploración general de datos para verificar los tipos de datos  con el fin de comprender de las características y el esquema de datos  y si es posible solucionar problemas básicos relacionados. realizar

- Descripción general de los datos
- Unificar la forma como se representan los valores Nulos
- Convertir los datos en su tipo correcto (numéricos, categóricos, booleanos, fechas, etc) y corrección de los datos si es necesario, para eu cada columna tenga un tipo de dato uniforme.
- Almacenar el dataset final en un formato adecuado como `.parquet`

# Entregables

Notebook con la exploración general de los datos y los pasos descritos anteriormente

Se debe realizar un Pull request para ingresar el notebook a la rama`main` para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD


## 📚 Import  libraries

In [1]:
# base libraries for data science
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa

## 💾 Load data

In [4]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data" / "01_raw"
corazon_df = pd.read_csv(DATA_DIR / "corazon_final_raw.csv", low_memory=False)

## 📊 Data description

In [ ]:
corazon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   str    
 1   sex         2969 non-null   str    
 2   chest_pain  2947 non-null   str    
 3   rest_bp     2949 non-null   str    
 4   chol        2945 non-null   str    
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   str    
 7   max_hr      2859 non-null   str    
 8   exang       2879 non-null   str    
 9   old_peak    2880 non-null   str    
 10  slope       2879 non-null   str    
 11  ca          2868 non-null   str    
 12  thal        2904 non-null   str    
 13  disease     2924 non-null   str    
dtypes: float64(1), str(13)
memory usage: 506.9 KB


In [9]:
corazon_df.sample(10)

,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
1254,71,Female,nontypical,160,302,0.0,normal,162,0,0.4,1,2.0,normal,0
3029,38,Male,nonanginal,138,175,0.0,normal,173,0,0.0,1,NaN,normal,0
2848,63,Female,asymptomatic,150,407,0.0,left ventricular hypertrophy,154,0,4.0,2,3.0,reversable,1
1774,57,Male,nontypical,124,261,0.0,normal,141,0,0.3,1,0.0,reversable,1
11,56,Female,nontypical,140,294,0.0,left ventricular hypertrophy,153,0,1.3,2,0.0,normal,0
1931,43,Female,asymptomatic,132,341,1.0,left ventricular hypertrophy,136,1,3.0,2,0.0,reversable,1
846,41,Male,nontypical,110,235,0.0,normal,153,0,0.0,1,0.0,normal,0
2602,43,Male,nonanginal,130,315,0.0,normal,162,0,1.9,1,1.0,normal,0
1346,43,Female,nonanginal,122,213,0.0,normal,165,0,0.2,2,0.0,normal,0
1362,52,Male,typical,152,298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,reversable,0


In [11]:
corazon_df.describe(include="all")

,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
count,3000,2969,2947,2949,2945,2933.000000,2837,2859,2879,2880,2879,2868,2904,2924
unique,43,5,7,52,154,NaN,8,92,4,42,4,5,7,7
top,58,Male,asymptomatic,120,204,NaN,normal,162,0,0.0,1,0.0,normal,0
freq,187,2017,1394,361,59,NaN,1409,103,1929,944,1353,1684,1599,1576
mean,NaN,NaN,NaN,NaN,NaN,0.148312,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,0.355470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 🧹 Unificar representación de valores nulos

In [10]:
for col in corazon_df.select_dtypes(include="object").columns:
    print(f"--- {col} ---")
    print(corazon_df[col].unique())
    print()

--- age ---
<ArrowStringArray>
[    '63',     '67',     '37',     '41',     '56',     '62',     '57',
     '53',     '44',     '52',     '48',     '54',     '49',     '64',
     '58',     '60',     '50',     '66',     '43',     '40',     '69',
     '59',     '42',     '55',     '61',     '65',     '71',     '51',
     '46',     '45',     '39',     '68',     '47',     '34',     '35',
     '29',     '70',     '77',     '38',     '74',     '76', 'fggfds',
    'sdg',      nan]
Length: 44, dtype: str

--- sex ---
<ArrowStringArray>
['Male', 'Female', '2345', '45', '765', nan]
Length: 6, dtype: str

--- chest_pain ---
<ArrowStringArray>
[     'typical', 'asymptomatic',   'nonanginal',   'nontypical',
         '2345',         '2435',         '3456',            nan]
Length: 8, dtype: str

--- rest_bp ---
<ArrowStringArray>
[ '145',  '160',  '120',  '130',  '140',  '172',  '150',  '110',  '132',
  '117',  '135',  '112',  '105',  '124',  '125',  '142',  '128',  '170',
  '155',  '104',  '180',  '

/tmp/ipykernel_245509/4057101631.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in corazon_df.select_dtypes(include="object").columns:


In [12]:
numeric_cols = ["age", "rest_bp", "chol", "max_hr", "old_peak"]

for col in numeric_cols:
    corazon_df[col] = pd.to_numeric(corazon_df[col], errors="coerce")

In [14]:
valid_categories = {
    "sex": ["Male", "Female"],
    "chest_pain": ["typical", "asymptomatic", "nonanginal", "nontypical"],
    "rest_ecg": ["normal", "left ventricular hypertrophy", "ST-T wave abnormality"],
    "thal": ["normal", "fixed", "reversable"],
    "fbs": ["0.0", "1.0"],
    "exang": ["0", "1"],
    "slope": ["1", "2", "3"],
    "ca": ["0.0", "1.0", "2.0", "3.0"],
    "disease": ["0", "1"],
}

for col, valid_values in valid_categories.items():
    corazon_df[col] = corazon_df[col].astype("string").str.strip()
    corazon_df[col] = corazon_df[col].where(corazon_df[col].isin(valid_values), np.nan)

In [15]:
corazon_df.isna().sum()

age            32
sex            64
chest_pain     86
rest_bp        83
chol           87
fbs            97
rest_ecg      198
max_hr        172
exang         153
old_peak      152
slope         152
ca            163
thal          130
disease       111
dtype: int64

## 🗑️ Remove columns

Se evaluó cada columna bajo dos criterios: (1) porcentaje de valores faltantes, y (2) si se trata de un identificador sin valor predictivo real.

En cuanto a valores faltantes, la columna con mayor proporción de nulos es `rest_ecg` con 198 de 3030 registros (6.5%), muy por debajo de un umbral típico de eliminación (30-50%).

En cuanto a identificadores sin valor predictivo, ninguna de las 13 variables corresponde a este caso; todas son variables clínicas respaldadas por literatura médica como factores relacionados con enfermedad cardíaca.

Por lo tanto, se conservan todas las columnas del dataset.

## 🗂️ Clasificación de variables

**Categóricas nominales** (categorías distintas, sin orden lógico entre ellas):
- `sex`: Male, Female
- `chest_pain`: typical, asymptomatic, nonanginal, nontypical
- `rest_ecg`: normal, left ventricular hypertrophy, ST-T wave abnormality
- `thal`: normal, fixed, reversable

**Categórica ordinal** (categorías con un orden clínico reconocido):
- `slope`: 1 (ascendente), 2 (plana), 3 (descendente) — de menor a mayor riesgo asociado

**Numéricas continuas** (miden una magnitud física, admiten decimales con sentido real):
- `age`: edad en años
- `rest_bp`: presión arterial en reposo (mm Hg)
- `chol`: colesterol (mg/dl)
- `old_peak`: depresión del segmento ST

**Numéricas discretas** (cuentan eventos u objetos completos):
- `max_hr`: frecuencia cardíaca máxima (latidos por minuto)
- `ca`: número de vasos principales afectados

**Booleanas** (dos categorías: verdadero/falso):
- `fbs`: glucemia en ayunas > 120 mg/dl
- `exang`: angina inducida por ejercicio
- `disease`: variable objetivo, indica si el paciente tiene la enfermedad

## 🔢 Convertir los datos en su tipo correcto

### Categorical variables

In [16]:
categorical_cols = ["sex", "chest_pain", "rest_ecg", "thal"]
corazon_df[categorical_cols] = corazon_df[categorical_cols].astype("category")

In [17]:
corazon_df["slope"] = pd.Categorical(
    corazon_df["slope"], categories=["1", "2", "3"], ordered=True
)

### Numerical variables

In [18]:
continuous_cols = ["age", "rest_bp", "chol", "old_peak"]
corazon_df[continuous_cols] = corazon_df[continuous_cols].astype("float")

In [19]:
corazon_df["max_hr"] = corazon_df["max_hr"].astype("Int16")
corazon_df["ca"] = pd.to_numeric(corazon_df["ca"], errors="coerce").astype("Int8")

### Boolean variables

In [20]:
bool_map = {"0": False, "1": True, "0.0": False, "1.0": True}
bool_cols = ["fbs", "exang", "disease"]

for col in bool_cols:
    corazon_df[col] = corazon_df[col].map(bool_map).astype("boolean")

In [21]:
corazon_df.dtypes

age            float64
sex           category
chest_pain    category
rest_bp        float64
chol           float64
fbs            boolean
rest_ecg      category
max_hr           Int16
exang          boolean
old_peak       float64
slope         category
ca                Int8
thal          category
disease        boolean
dtype: object

## 💾 Save dataframe with data types

In [22]:
INTERMEDIATE_DIR = Path.cwd().resolve().parents[1] / "data" / "02_intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

file_path = INTERMEDIATE_DIR / "corazon_clean.parquet"
corazon_df.to_parquet(file_path, index=False)

## 📊 Analysis of Results and Conclusions 

A partir de la exploración inicial de los 3030 registros del dataset, se identificaron los siguientes hallazgos:

- **Calidad de los datos**: se encontraron valores inválidos (texto sin sentido y números fuera de rango) en prácticamente todas las columnas, probablemente introducidos intencionalmente como parte del ejercicio. Estos valores fueron identificados y convertidos a nulos, ya que aceptarlos tal cual habría distorsionado cualquier análisis o modelo posterior.

- **Valores faltantes**: después de la limpieza, cada columna presenta entre 1% y 6.5% de valores faltantes, siendo `rest_ecg` la más afectada. Ninguna columna alcanza un nivel de valores faltantes lo suficientemente alto como para justificar su eliminación (se evaluó frente a un umbral de referencia de 30-50%).

